# 设备 Log 标准化与良率洞察分析 Notebook

本 Notebook 用于处理半导体镀银设备导出的原始 Log 数据，将设备导出的宽表结构转换为统一口径的标准化分析底表，并在标准化数据基础上开展特征选择、EDA、异常检测、良率影响分析和 AI 洞察。

本 Notebook 的核心目标不是简单读取 Excel，而是将设备原始日志转化为后续分析可追溯、可解释、可复用的数据资产。

最终标准化输出文件为：`data/processed_equipment_log_standardized.xlsx`。

后续所有分析步骤都应基于该标准化文件展开，而不是直接依赖原始设备 Log。


## 步骤 1：环境与依赖说明

### 本步骤目标
说明本 Notebook 的运行依赖与参数配置方式。

### 输入
Notebook 内置配置项（原始文件名、输出文件名、开关变量）。

### 输出
后续步骤统一使用的配置参数。


In [ ]:
# =========================
# 配置区
# =========================
RAW_DATA_FILENAME = "M04.xlsx"
RAW_SHEET_NAME = None

OUTPUT_DIR_NAME = "已处理"

OUTPUT_WIDE_FILENAME = "processed_equipment_log_standardized_wide.xlsx"
OUTPUT_LONG_FILENAME = "processed_equipment_log_line_state_long.xlsx"
OUTPUT_DEVICE_STATE_FILENAME = "processed_equipment_log_device_state.xlsx"
OUTPUT_FIELD_MAPPING_FILENAME = "processed_equipment_log_field_mapping.xlsx"
OUTPUT_FIELD_PROFILE_FILENAME = "processed_equipment_log_field_profile.xlsx"

CREATE_ONOFF_NUMERIC_COLUMNS = True


In [ ]:
# 中文字体初始化（用于 Matplotlib 图表）
from pathlib import Path 
import matplotlib as mpl
from matplotlib import font_manager
import warnings

# 可选：隐藏缺字形告警，避免演示时出现红色 warning
warnings.filterwarnings("ignore", category=UserWarning, message=r"Glyph.*missing from font")

# 按优先级尝试字体：PingFang（macOS）-> STHeiti -> Hiragino Sans GB -> Arial Unicode
font_candidates = [
    ("PingFang SC", [
        "/System/Library/Fonts/PingFang.ttc",
    ]),
    ("STHeiti", [
        "/System/Library/Fonts/STHeiti Light.ttc",
        "/System/Library/Fonts/STHeiti Medium.ttc",
    ]),
    ("Hiragino Sans GB", [
        "/System/Library/Fonts/Hiragino Sans GB.ttc",
    ]),
    ("Arial Unicode MS", [
        "/Library/Fonts/Arial Unicode.ttf",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/System/Library/Fonts/Supplemental/Arial Unicode MS.ttf",
    ]),
]

selected_font = None
for font_name, font_paths in font_candidates:
    for font_path in font_paths:
        p = Path(font_path)
        if p.exists():
            try:
                font_manager.fontManager.addfont(str(p))
                selected_font = font_name
                break
            except Exception as e:
                print(f"加载字体失败 {font_path}: {e}")
    if selected_font:
        break

if selected_font:
    mpl.rcParams["font.family"] = selected_font
    mpl.rcParams["font.sans-serif"] = [selected_font]
    mpl.rcParams["axes.unicode_minus"] = False
    print(f"Matplotlib 中文字体已设置为: {selected_font}")
else:
    print("未找到可用中文字体（PingFang/STHeiti/Hiragino Sans GB/Arial Unicode）。图表可能出现中文缺字。")



## 步骤 2：路径配置与文件检查

### 本步骤目标
统一管理输入/输出路径，并确保输出目录存在。

### 后续用途
所有数据读写都通过路径变量完成，避免硬编码路径。


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / OUTPUT_DIR_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE_PATH = DATA_DIR / RAW_DATA_FILENAME  # 原始设备 Log 路径
WIDE_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_WIDE_FILENAME
LONG_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_LONG_FILENAME
PROCESSED_DATA_PATH = DATA_DIR / "processed_equipment_log_standardized.xlsx"  # 后续统一分析输入
DEVICE_STATE_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_DEVICE_STATE_FILENAME
FIELD_MAPPING_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FIELD_MAPPING_FILENAME
FIELD_PROFILE_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FIELD_PROFILE_FILENAME

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw file path: {RAW_FILE_PATH}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
def make_unique_columns(cols):
    counters = {}
    unique_cols = []
    raw_to_unique = []
    duplicate_summary = []
    for col in cols:
        col = str(col)
        if col not in counters:
            counters[col] = 0
            unique_col = col
        else:
            counters[col] += 1
            unique_col = f"{col}_{counters[col]}"
        unique_cols.append(unique_col)
        raw_to_unique.append((col, unique_col))
    for col, count in counters.items():
        if count > 0:
            duplicate_summary.append({"source_col": col, "duplicate_count": count + 1})
    return unique_cols, raw_to_unique, pd.DataFrame(duplicate_summary)

def strip_line_prefix(col_name):
    return re.sub(r"^L[12][\s_-]+", "", col_name).strip()

def make_unique_names(names):
    counters = {}
    out = []
    for n in names:
        if n not in counters:
            counters[n] = 0
            out.append(n)
        else:
            counters[n] += 1
            out.append(f"{n}_{counters[n]}")
    return out

def safe_device_name(base_name, occupied):
    candidate = f"device_{base_name}"
    i = 0
    while candidate in occupied:
        i += 1
        candidate = f"device_{base_name}_{i}"
    occupied.add(candidate)
    return candidate


## 步骤 3：原始数据结构理解

### 本步骤目标
读取原始设备 Log 并了解其行列规模。

### 为什么不能直接分析原始数据
- L1 / L2 字段横向展开，同一工艺参数被拆成两套变量，难以比较。
- 可能存在重复列名，pandas 自动改名后容易造成字段语义混淆。
- 字段名携带 L1 / L2 前缀，后续特征选择可能把同义字段误判为不同变量。
- 原始数据是设备导出记录格式，不是分析建模格式。
- 缺少统一来源标识，不利于追踪记录来自 L1 还是 L2。

**结论：原始设备 Log 是设备记录格式，不是分析建模格式，因此需要先做标准化转换，再进行后续分析。**


In [ ]:
# 读取原始设备 Log（设备导出宽表）。
# 该数据用于标准化转换，不直接作为后续建模输入。
if RAW_SHEET_NAME is None:
    df_raw = pd.read_excel(RAW_FILE_PATH)
    sheet_info = "first sheet (default)"
else:
    df_raw = pd.read_excel(RAW_FILE_PATH, sheet_name=RAW_SHEET_NAME)
    sheet_info = RAW_SHEET_NAME
print(f"原始文件路径: {RAW_FILE_PATH}")
print(f"sheet 名: {sheet_info}")
print(f"原始数据行数: {len(df_raw)}")
print(f"原始数据列数: {df_raw.shape[1]}")


In [ ]:
original_cols = [str(c) for c in df_raw.columns]
unique_cols, raw_to_unique, duplicate_summary_df = make_unique_columns(original_cols)
df = df_raw.copy()
df.columns = unique_cols
print(f"重复列（按原始列名统计）数量: {len(duplicate_summary_df)}")
if len(duplicate_summary_df) > 0:
    display(duplicate_summary_df)
raw_unique_df = pd.DataFrame(raw_to_unique, columns=["source_col", "source_col_unique"])


In [ ]:
lower_map = {c.lower(): c for c in df.columns}
date_col = lower_map.get("date")
time_col = lower_map.get("time")
if date_col is not None and time_col is not None:
    timestamp_series = pd.to_datetime(df[date_col].astype(str).str.strip() + " " + df[time_col].astype(str).str.strip(), errors="coerce")
else:
    ts_candidates = [c for c in df.columns if c.lower() in ["timestamp", "datetime", "date_time"]]
    if not ts_candidates:
        raise ValueError("未找到可用时间字段：请至少包含 date + Time 或 timestamp 列。")
    timestamp_series = pd.to_datetime(df[ts_candidates[0]], errors="coerce")
if "timestamp" in df.columns:
    df = df.drop(columns=["timestamp"])
df.insert(0, "timestamp", timestamp_series)
df = df.sort_values("timestamp", ascending=True).reset_index(drop=True)
invalid_ts_count = df["timestamp"].isna().sum()
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"无法解析 timestamp 的记录数: {invalid_ts_count}")
print("采样记录:")
display(df[["timestamp"] + ([date_col] if date_col else []) + ([time_col] if time_col else [])].head())


## 步骤 4：识别 Time/L1/L2 字段区域

### 处理逻辑
识别时间字段、L1 字段、L2 字段与设备公共状态字段，为后续标准化转换做准备。


In [ ]:
# 识别 L1/L2 字段集合与公共字段集合，用于后续宽转长。
time_cols = [c for c in ["timestamp", date_col, time_col] if c is not None and c in df.columns]
l1_pattern = re.compile(r"^L1[\s_-]+")
l2_pattern = re.compile(r"^L2[\s_-]+")
l1_cols = [c for c in df.columns if l1_pattern.search(c)]
l2_cols = [c for c in df.columns if l2_pattern.search(c)]
device_state_cols = [c for c in df.columns if c not in set(time_cols + l1_cols + l2_cols)]
print(f"L1 字段数量: {len(l1_cols)}")
print(f"L2 字段数量: {len(l2_cols)}")
print(f"设备共有状态字段数量: {len(device_state_cols)}")


In [ ]:
wide_df = df.copy()
wide_df.to_excel(WIDE_OUTPUT_PATH, index=False)
print(f"已输出标准化宽表: {WIDE_OUTPUT_PATH}")


In [ ]:
device_state_df = df[["timestamp"] + device_state_cols].copy()
occupied_names = set()
device_name_map = {}
for col in device_state_cols:
    mapped = safe_device_name(col, occupied_names)
    device_name_map[col] = mapped
if device_state_cols:
    device_state_df = device_state_df.rename(columns=device_name_map)
device_state_df.to_excel(DEVICE_STATE_OUTPUT_PATH, index=False)
print(f"已输出设备共有状态表: {DEVICE_STATE_OUTPUT_PATH}")


## 步骤 5：数据转换处理（L1/L2 宽表转标准化长表）

### 核心链路
原始设备 Log Excel → 读取原始列名 → 处理重复列 → 识别 Time/L1/L2 字段区域 → 拆分 L1 子表和 L2 子表 → 去除 L1/L2 前缀并统一字段名 → 新增 Line 字段 → 合并为标准化长表。

### 关键概念
L1 xxx 和 L2 xxx 原本是两套横向字段；标准化后，L1/L2 不再体现在列名前缀中，而是通过 `line` 字段体现。


In [ ]:
# 将 L1/L2 从列名前缀转换为行级字段 line，便于横向比较与异常检测。
def build_line_long(df_input, line_cols, line_name):
    part = df_input[["timestamp"] + ([date_col] if date_col else []) + ([time_col] if time_col else []) + line_cols].copy()
    base_names = [strip_line_prefix(c) for c in line_cols]
    base_names = make_unique_names(base_names)
    rename_map = {old: new for old, new in zip(line_cols, base_names)}
    part = part.rename(columns=rename_map)
    part.insert(1, "line", line_name)
    if CREATE_ONOFF_NUMERIC_COLUMNS:
        for c in list(part.columns):
            if c.endswith("OnOff") or " OnOff" in c or "_OnOff" in c:
                part[f"{c}__num"] = part[c].map({"On": 1, "Off": 0})
    return part, rename_map
l1_long, l1_rename = build_line_long(df, l1_cols, "L1")
l2_long, l2_rename = build_line_long(df, l2_cols, "L2")
line_state_long = pd.concat([l1_long, l2_long], ignore_index=True)
line_state_long = line_state_long.merge(device_state_df, on="timestamp", how="left")
line_state_long = line_state_long.sort_values(["timestamp", "line"], ascending=True).reset_index(drop=True)
line_state_long.to_excel(LONG_OUTPUT_PATH, index=False)
print(f"已输出产线状态长表: {LONG_OUTPUT_PATH}")
print(f"长表行数: {len(line_state_long)} (原始行数: {len(df)})")


## 特殊字段处理说明

1. `L2 Strike OnOff`：仅出现在 L2 区域。标准化后保留该字段，L1 记录填充空值，确保统一字段结构且不丢失 L2 独有信号。
2. `L2 Oven(1) TC`：L2 中存在重复字段。本阶段进行唯一化命名并保留，不贸然删除，后续结合工艺语义再判断是否合并。

数据清洗的目标不是“表面更干净”，而是在不误删潜在业务信号前提下提升可分析性。


In [ ]:
mapping_rows = []
for src, src_unique in raw_to_unique:
    mapping_rows.append({"source_col": src, "source_col_unique": src_unique, "target_col": src_unique, "group": "wide", "line": "", "base_name": "", "target_table": "standardized_wide", "note": "标准化宽表保留字段"})
for c in l1_cols:
    mapping_rows.append({"source_col": c, "source_col_unique": c, "target_col": l1_rename[c], "group": "line_state", "line": "L1", "base_name": l1_rename[c], "target_table": "line_state_long", "note": "L1 产线参数"})
for c in l2_cols:
    mapping_rows.append({"source_col": c, "source_col_unique": c, "target_col": l2_rename[c], "group": "line_state", "line": "L2", "base_name": l2_rename[c], "target_table": "line_state_long", "note": "L2 产线参数"})
for src, tgt in device_name_map.items():
    mapping_rows.append({"source_col": src, "source_col_unique": src, "target_col": tgt, "group": "device_state", "line": "device", "base_name": src, "target_table": "device_state / line_state_long", "note": "设备共有状态参数"})
field_mapping_df = pd.DataFrame(mapping_rows)
field_mapping_df.to_excel(FIELD_MAPPING_OUTPUT_PATH, index=False)
print(f"已输出字段映射表: {FIELD_MAPPING_OUTPUT_PATH}")


In [ ]:
def profile_dataframe(df_input, group_name, line_name="", base_name_map=None):
    rows = []
    for c in df_input.columns:
        s = df_input[c]
        s_non_na = s.dropna()
        is_numeric = pd.api.types.is_numeric_dtype(s)
        is_onoff = s_non_na.astype(str).isin(["On", "Off"]).all() if len(s_non_na) else False
        rows.append({
            "field_name": c,
            "group": group_name,
            "line": line_name,
            "base_name": base_name_map.get(c, c) if base_name_map else c,
            "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()),
            "missing_ratio": float(s.isna().mean()),
            "unique_count": int(s.nunique(dropna=True)),
            "is_all_missing": bool(s.isna().all()),
            "is_numeric": bool(is_numeric),
            "is_onoff": bool(is_onoff),
            "min": s.min() if is_numeric else np.nan,
            "max": s.max() if is_numeric else np.nan,
            "mean": s.mean() if is_numeric else np.nan,
            "std": s.std() if is_numeric else np.nan,
        })
    return rows
profile_rows = []
profile_rows += profile_dataframe(wide_df, "wide")
profile_rows += profile_dataframe(line_state_long, "line_state_long")
profile_rows += profile_dataframe(device_state_df, "device_state")
field_profile_df = pd.DataFrame(profile_rows)
field_profile_df.to_excel(FIELD_PROFILE_OUTPUT_PATH, index=False)
print(f"已输出字段质量画像表: {FIELD_PROFILE_OUTPUT_PATH}")


In [ ]:
# 同步输出标准化主文件，作为后续分析统一输入源。
combined_for_analysis = line_state_long.copy()
combined_for_analysis.to_excel(PROCESSED_DATA_PATH, index=False)
print("标准化分析底表路径:", PROCESSED_DATA_PATH)
print("标准化宽表路径:", WIDE_OUTPUT_PATH)
print("产线状态长表路径:", LONG_OUTPUT_PATH)
print("设备共有状态表路径:", DEVICE_STATE_OUTPUT_PATH)
print("字段映射表路径:", FIELD_MAPPING_OUTPUT_PATH)
print("字段质量画像表路径:", FIELD_PROFILE_OUTPUT_PATH)
print("\n标准化宽表维度:", wide_df.shape)
print("产线状态长表维度:", line_state_long.shape)
print("设备共有状态表维度:", device_state_df.shape)
display(wide_df.head())
display(line_state_long.head())
display(device_state_df.head())
display(field_mapping_df.head())
display(field_profile_df.head())


## 标准化数据文件的作用

从这一层开始，后续分析应统一基于：`data/processed_equipment_log_standardized.xlsx`。

好处包括：避免重复处理原始文件、统一分析口径、便于复核转换结果、支持项目交接与客户演示、可作为 MES/BI/AI 平台中间层。


## 数据血缘图：从原始设备 Log 到良率洞察

本 Notebook 不直接基于原始设备 Log 进行后续分析，而是先将设备导出的宽表数据转换为统一口径的标准化分析底表。

数据血缘图用于说明：

1. 原始数据经过了哪些处理步骤；
2. L1 / L2 字段如何被统一为可比较的数据结构；
3. 标准化文件如何成为后续分析的唯一输入；
4. 后续 EDA、异常检测、良率影响分析和 AI 洞察如何基于同一份可信数据展开。

这样可以保证分析结果具备可追溯性、可解释性和可复用性。


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    raise ImportError("请先安装 networkx：pip install networkx")

nodes = {
    "原始设备 Log Excel": "数据源",
    "重复列处理": "处理步骤",
    "L1 / L2 字段拆分": "处理步骤",
    "统一字段命名": "处理步骤",
    "新增 Line 字段": "处理步骤",
    "processed_equipment_log_standardized.xlsx": "标准化数据资产",
    "特征选择与清洗层": "处理步骤",
    "EDA / 趋势分析": "分析应用",
    "异常检测": "分析应用",
    "良率影响分析 / AI 洞察": "分析应用",
}

edges = [
    ("原始设备 Log Excel", "重复列处理"),
    ("重复列处理", "L1 / L2 字段拆分"),
    ("L1 / L2 字段拆分", "统一字段命名"),
    ("统一字段命名", "新增 Line 字段"),
    ("新增 Line 字段", "processed_equipment_log_standardized.xlsx"),
    ("processed_equipment_log_standardized.xlsx", "特征选择与清洗层"),
    ("特征选择与清洗层", "EDA / 趋势分析"),
    ("特征选择与清洗层", "异常检测"),
    ("特征选择与清洗层", "良率影响分析 / AI 洞察"),
]

node_colors = {
    "数据源": "#B3E5FC",
    "处理步骤": "#FFF3B0",
    "标准化数据资产": "#C8E6C9",
    "分析应用": "#F8BBD0",
}

G = nx.DiGraph()
for node, node_type in nodes.items():
    G.add_node(node, node_type=node_type)
G.add_edges_from(edges)

pos = {
    "原始设备 Log Excel": (0, 0),
    "重复列处理": (0, -1),
    "L1 / L2 字段拆分": (0, -2),
    "统一字段命名": (0, -3),
    "新增 Line 字段": (0, -4),
    "processed_equipment_log_standardized.xlsx": (0, -5),
    "特征选择与清洗层": (0, -6),
    "EDA / 趋势分析": (-2, -7),
    "异常检测": (0, -7),
    "良率影响分析 / AI 洞察": (2, -7),
}

plt.figure(figsize=(16, 10))
colors = [node_colors[G.nodes[n]["node_type"]] for n in G.nodes]
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=3200, edgecolors="#555555")
nx.draw_networkx_labels(G, pos, font_size=11)
nx.draw_networkx_edges(
    G,
    pos,
    arrows=True,
    arrowstyle='-|>',
    arrowsize=18,
    edge_color="#6D6D6D",
    width=2,
    connectionstyle="arc3,rad=0.02",
)

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=label, markerfacecolor=color, markersize=12)
    for label, color in node_colors.items()
]
plt.legend(handles=legend_handles, loc="upper left", frameon=False)
plt.title("数据血缘图：原始设备 Log 到标准化数据与良率洞察分析", fontsize=16, pad=20)
plt.axis("off")
plt.tight_layout()

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
lineage_img_path = output_dir / "data_lineage_overview.png"
plt.savefig(lineage_img_path, dpi=200, bbox_inches="tight")
print(f"数据血缘图已保存: {lineage_img_path}")
plt.show()



In [ ]:
from pathlib import Path
# 向上一级目录，定位到项目根目录
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_PATH = DATA_DIR / "processed_equipment_log_standardized.xlsx"
if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"未找到标准化后的数据文件：{PROCESSED_DATA_PATH}\n请先运行前面的数据标准化处理 Cell，生成 processed_equipment_log_standardized.xlsx。")
print(f"使用标准化后的数据文件进行特征选择与清洗：{PROCESSED_DATA_PATH}")

import pandas as pd
from IPython.display import display

lineage_desc = [
    {"步骤": 1, "节点": "原始设备 Log Excel", "类型": "数据源", "说明": "设备导出的原始宽表数据，包含 Time、L1 字段、L2 字段等", "输出": "原始 DataFrame"},
    {"步骤": 2, "节点": "重复列处理", "类型": "数据处理", "说明": "处理重复字段名，避免字段覆盖或歧义", "输出": "列名唯一化后的 DataFrame"},
    {"步骤": 3, "节点": "L1 / L2 字段拆分", "类型": "结构转换", "说明": "识别 L1 与 L2 两套设备/产品维度字段", "输出": "L1 子表、L2 子表"},
    {"步骤": 4, "节点": "统一字段命名", "类型": "字段标准化", "说明": "去除 L1 / L2 前缀，使同类工艺参数进入同一列", "输出": "统一字段口径"},
    {"步骤": 5, "节点": "新增 Line 字段", "类型": "业务标识", "说明": "通过 Line=L1/L2 保留原始来源信息", "输出": "标准化长表"},
    {"步骤": 6, "节点": "processed_equipment_log_standardized.xlsx", "类型": "标准化数据资产", "说明": "后续所有分析统一基于该文件，不再直接依赖原始 Excel", "输出": "标准化分析底表"},
    {"步骤": 7, "节点": "特征选择与清洗层", "类型": "分析准备", "说明": "筛选可分析字段，处理缺失、常数列、异常值等", "输出": "特征矩阵 / 清洗后数据"},
    {"步骤": 8, "节点": "EDA / 异常检测 / 良率影响分析 / AI 洞察", "类型": "分析应用", "说明": "基于标准化数据进行工艺理解、异常定位和洞察生成", "输出": "分析结论与客户演示内容"},
]

lineage_df = pd.DataFrame(lineage_desc)
display(lineage_df)



## 特征选择与清洗层

本层不是重新读取原始设备 Log，而是在标准化数据基础上构建可分析特征集。

从本步骤开始，所有 EDA、异常检测、良率影响分析与 AI 洞察都应统一基于 `data/processed_equipment_log_standardized.xlsx`。


## 总结：从设备 Log 到可解释良率洞察

本 Notebook 完成了从原始设备 Log 到标准化分析底表，再到良率洞察分析的链路沉淀。

核心产出：
1. 标准化数据文件：`data/processed_equipment_log_standardized.xlsx`
2. 数据血缘图：`outputs/data_lineage_overview.png`
3. 字段映射与字段画像结果
4. 可供后续 EDA、异常检测、良率影响分析与 AI 洞察复用的标准化输入

其价值在于构建了可追溯、可解释、可复用的数据资产流程。


## 第 1 次升级：MVP AI 生产状态回放洞察


In [ ]:
# ============================================================
# MVP AI 生产状态回放洞察配置
# ============================================================
ENABLE_AI_INSIGHT = True

# 可选：ollama / openai / custom_http / none
LLM_PROVIDER = "ollama"

# 本地 Ollama 配置
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma4:e4b"

# 未来云端模型预留配置
OPENAI_MODEL = "gpt-4.1-mini"
OPENAI_API_KEY_ENV = "OPENAI_API_KEY"

# 自定义 HTTP 大模型接口预留配置
CUSTOM_LLM_ENDPOINT = ""
CUSTOM_LLM_MODEL = ""
CUSTOM_LLM_API_KEY_ENV = ""

# 模型调用失败时，是否自动使用规则版报告
USE_FALLBACK_REPORT = True

# 是否保存 AI 洞察结果
SAVE_AI_INSIGHT_REPORT = True
AI_INSIGHT_OUTPUT_DIR_NAME = "ai_insights"

# 如果前文未定义输出目录变量，则使用默认值
PROCESSED_DIR_NAME = globals().get("OUTPUT_DIR_NAME", "已处理")
OUTPUT_LONG_FILENAME = globals().get("OUTPUT_LONG_FILENAME", "processed_equipment_log_line_state_long.xlsx")
OUTPUT_DEVICE_STATE_FILENAME = globals().get("OUTPUT_DEVICE_STATE_FILENAME", "processed_equipment_log_device_state.xlsx")
OUTPUT_FIELD_MAPPING_FILENAME = globals().get("OUTPUT_FIELD_MAPPING_FILENAME", "processed_equipment_log_field_mapping.xlsx")


In [ ]:
# ============================================================
# 回放窗口配置
# ============================================================
REPLAY_START_TIME = "2026-04-20 08:00:00"
REPLAY_END_TIME = "2026-04-20 10:00:00"

# 观察参数使用“产线长表中的 base_name 字段名”
SELECTED_REPLAY_PARAMS = [
    "ED(1) Current",
    "ED(1) Voltage",
    "Oven(1) TC",
]

# 设备共有状态参数（可选）
SELECTED_DEVICE_STATE_PARAMS = []
TOP_N_SUMMARY = 10


In [ ]:
import json
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / PROCESSED_DIR_NAME

line_state_long = pd.read_excel(PROCESSED_DIR / OUTPUT_LONG_FILENAME)
device_state_df = pd.read_excel(PROCESSED_DIR / OUTPUT_DEVICE_STATE_FILENAME)
field_mapping_df = pd.read_excel(PROCESSED_DIR / OUTPUT_FIELD_MAPPING_FILENAME)

required_line_cols = {"timestamp", "line"}
required_device_cols = {"timestamp"}
if not required_line_cols.issubset(line_state_long.columns):
    raise ValueError(f"line_state_long 缺少必要字段: {required_line_cols - set(line_state_long.columns)}")
if not required_device_cols.issubset(device_state_df.columns):
    raise ValueError(f"device_state_df 缺少必要字段: {required_device_cols - set(device_state_df.columns)}")

line_state_long["timestamp"] = pd.to_datetime(line_state_long["timestamp"], errors="coerce")
device_state_df["timestamp"] = pd.to_datetime(device_state_df["timestamp"], errors="coerce")

line_values = sorted([x for x in line_state_long["line"].dropna().unique().tolist()])
missing_lines = {"L1", "L2"} - set(line_values)
if missing_lines:
    print(f"[warning] line 字段中未找到: {sorted(missing_lines)}")

print("产线状态长表维度:", line_state_long.shape)
print("设备共有状态表维度:", device_state_df.shape)
print("字段映射表维度:", field_mapping_df.shape)
print("时间范围(line_state_long):", line_state_long["timestamp"].min(), "->", line_state_long["timestamp"].max())
print("时间范围(device_state_df):", device_state_df["timestamp"].min(), "->", device_state_df["timestamp"].max())
print("line 取值:", line_values)


In [ ]:
def filter_replay_window(line_state_long, device_state_df, start_time, end_time):
    start_ts = pd.to_datetime(start_time)
    end_ts = pd.to_datetime(end_time)
    if pd.isna(start_ts) or pd.isna(end_ts):
        raise ValueError("回放时间配置无效，请检查 REPLAY_START_TIME / REPLAY_END_TIME")
    if end_ts <= start_ts:
        raise ValueError("REPLAY_END_TIME 必须大于 REPLAY_START_TIME")

    line_window = line_state_long[(line_state_long["timestamp"] >= start_ts) & (line_state_long["timestamp"] <= end_ts)].copy()
    device_window = device_state_df[(device_state_df["timestamp"] >= start_ts) & (device_state_df["timestamp"] <= end_ts)].copy()

    if line_window.empty:
        raise ValueError(f"回放窗口内无产线状态数据: {start_ts} ~ {end_ts}")
    if device_window.empty:
        raise ValueError(f"回放窗口内无设备状态数据: {start_ts} ~ {end_ts}")

    print("回放窗口开始时间:", start_ts)
    print("回放窗口结束时间:", end_ts)
    print("产线状态记录数:", len(line_window))
    print("设备状态记录数:", len(device_window))
    print("L1 记录数:", (line_window["line"] == "L1").sum())
    print("L2 记录数:", (line_window["line"] == "L2").sum())
    return line_window, device_window


def build_replay_summary(line_window, device_window, selected_params=None, selected_device_params=None, top_n=10):
    all_base_params = line_window["base_name"].dropna().unique().tolist() if "base_name" in line_window.columns else []
    selected_params = selected_params or []
    valid_selected = [p for p in selected_params if p in all_base_params]
    invalid_selected = [p for p in selected_params if p not in all_base_params]
    if invalid_selected:
        print(f"[warning] 以下产线参数不存在，已忽略: {invalid_selected}")

    numeric_line = line_window[pd.to_numeric(line_window.get("value"), errors="coerce").notna()].copy() if "value" in line_window.columns else pd.DataFrame()
    if not valid_selected:
        auto_params = numeric_line["base_name"].value_counts().head(max(3, min(top_n, 10))).index.tolist() if not numeric_line.empty else []
        valid_selected = auto_params
        print(f"[info] 自动选择产线参数: {valid_selected}")

    def metric_for_param(param):
        sub = line_window[line_window["base_name"] == param].copy()
        sub["num_value"] = pd.to_numeric(sub["value"], errors="coerce")
        sub = sub.dropna(subset=["num_value", "timestamp", "line"])
        if sub.empty:
            return None
        pivot = sub.pivot_table(index="timestamp", columns="line", values="num_value", aggfunc="mean")
        if "L1" not in pivot.columns or "L2" not in pivot.columns:
            print(f"[warning] 参数 {param} 无法完成 L1/L2 对齐")
            return None
        aligned = pivot[["L1", "L2"]].dropna()
        if aligned.empty:
            print(f"[warning] 参数 {param} 对齐后无可用样本")
            return None
        diff = aligned["L2"] - aligned["L1"]
        return {
            "parameter": param,
            "l1_mean": float(aligned["L1"].mean()),
            "l2_mean": float(aligned["L2"].mean()),
            "l1_std": float(aligned["L1"].std(ddof=0)),
            "l2_std": float(aligned["L2"].std(ddof=0)),
            "l2_minus_l1_mean": float(diff.mean()),
            "l2_minus_l1_max_abs": float(diff.abs().max()),
            "diff_std": float(diff.std(ddof=0)),
            "l1_l2_corr": float(aligned["L1"].corr(aligned["L2"])) if len(aligned) > 1 else None,
        }

    all_numeric_params = line_window["base_name"].dropna().unique().tolist() if "base_name" in line_window.columns else []
    all_metrics = [m for m in (metric_for_param(p) for p in all_numeric_params) if m]
    line_comparison = [m for m in (metric_for_param(p) for p in valid_selected) if m]
    top_line_diff_parameters = sorted(all_metrics, key=lambda x: (abs(x["l2_minus_l1_mean"]), x["diff_std"], x["l2_minus_l1_max_abs"]), reverse=True)[:top_n]

    top_line_volatile = {"L1": [], "L2": []}
    for line_name in ["L1", "L2"]:
        sub_line = line_window[line_window["line"] == line_name].copy()
        vol = []
        for param, g in sub_line.groupby("base_name"):
            values = pd.to_numeric(g["value"], errors="coerce").dropna()
            if len(values) >= 2:
                vol.append({"parameter": param, "std": float(values.std(ddof=0)), "mean": float(values.mean())})
        top_line_volatile[line_name] = sorted(vol, key=lambda x: x["std"], reverse=True)[:top_n]

    numeric_device_cols = [c for c in device_window.columns if c != "timestamp" and pd.api.types.is_numeric_dtype(device_window[c])]
    selected_device_params = selected_device_params or []
    valid_device = [c for c in selected_device_params if c in numeric_device_cols]
    invalid_device = [c for c in selected_device_params if c not in numeric_device_cols]
    if invalid_device:
        print(f"[warning] 以下设备共有状态参数不存在或非数值型，已忽略: {invalid_device}")
    if not valid_device:
        std_rank = device_window[numeric_device_cols].std(numeric_only=True).sort_values(ascending=False)
        valid_device = std_rank.head(max(3, min(top_n, 10))).index.tolist()
        print(f"[info] 自动选择设备共有状态参数: {valid_device}")

    device_state_summary = []
    for c in valid_device:
        series = pd.to_numeric(device_window[c], errors="coerce")
        if series.notna().sum() == 0:
            continue
        device_state_summary.append({
            "parameter": c,
            "mean": float(series.mean()),
            "std": float(series.std(ddof=0)),
            "min": float(series.min()),
            "max": float(series.max()),
            "range": float(series.max() - series.min()),
            "missing_ratio": float(series.isna().mean()),
        })

    top_device_volatile_parameters = sorted(device_state_summary, key=lambda x: x["std"], reverse=True)[:top_n]

    suggested_parameters = list(dict.fromkeys([x["parameter"] for x in top_line_diff_parameters[:5]] +
                                              [x["parameter"] for x in top_line_volatile["L1"][:3]] +
                                              [x["parameter"] for x in top_line_volatile["L2"][:3]]))
    suggested_device = [x["parameter"] for x in top_device_volatile_parameters[:5]]

    replay_summary = {
        "time_window": {
            "start": str(line_window["timestamp"].min()),
            "end": str(line_window["timestamp"].max()),
            "line_record_count": int(len(line_window)),
            "device_record_count": int(len(device_window)),
            "l1_record_count": int((line_window["line"] == "L1").sum()),
            "l2_record_count": int((line_window["line"] == "L2").sum()),
        },
        "selected_parameters": valid_selected,
        "selected_device_state_parameters": valid_device,
        "line_comparison": line_comparison,
        "top_line_diff_parameters": top_line_diff_parameters,
        "top_line_volatile_parameters": top_line_volatile,
        "device_state_summary": device_state_summary,
        "top_device_volatile_parameters": top_device_volatile_parameters,
        "next_replay_candidates": {
            "suggested_parameters": suggested_parameters,
            "suggested_device_parameters": suggested_device,
            "suggested_views": ["L1/L2 同名参数趋势图", "L2-L1 差值图", "设备共有状态趋势图", "标准化趋势图"],
        },
    }
    return replay_summary


In [ ]:
def build_ai_prompt(replay_summary):
    summary_text = json.dumps(replay_summary, ensure_ascii=False, indent=2, default=str)
    return f"""
你是生产状态回放分析助手。
你只能基于 replay_summary 进行分析，不要编造未提供的数据。
不要直接说某参数导致良率下降；不要把相关性、同步变化说成因果关系。

请严格按以下结构输出：
## AI 生产状态回放洞察
### 1. 当前窗口概览
### 2. L1 / L2 同步性观察
### 3. 设备共有状态观察
### 4. 值得关注的参数
### 5. 下一步回放建议
### 6. 不确定性说明

输出风格：客观、克制、面向工程师、给出可执行下一步建议。

replay_summary 如下：
{summary_text[:12000]}
""".strip()


def call_ollama(prompt: str) -> str:
    url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/chat"
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": "你是生产状态回放分析助手。必须基于给定结构化摘要进行客观分析，不要编造数据，不要直接下因果结论。"},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
    }
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    return data.get("message", {}).get("content", "")


def call_llm(prompt: str):
    if not ENABLE_AI_INSIGHT or LLM_PROVIDER == "none":
        return None
    if LLM_PROVIDER == "ollama":
        return call_ollama(prompt)
    if LLM_PROVIDER == "openai":
        raise NotImplementedError("OpenAI provider 尚未在本 MVP 中实现，请使用 ollama。")
    if LLM_PROVIDER == "custom_http":
        raise NotImplementedError("custom_http provider 尚未在本 MVP 中实现。")
    raise ValueError(f"未知 LLM_PROVIDER: {LLM_PROVIDER}")


def build_fallback_report(replay_summary, error_message=None):
    tw = replay_summary["time_window"]
    top_diff = replay_summary.get("top_line_diff_parameters", [])[:5]
    l1_vol = replay_summary.get("top_line_volatile_parameters", {}).get("L1", [])[:5]
    l2_vol = replay_summary.get("top_line_volatile_parameters", {}).get("L2", [])[:5]
    top_device = replay_summary.get("top_device_volatile_parameters", [])[:5]
    nxt = replay_summary.get("next_replay_candidates", {})

    lines = ["## 生产状态回放洞察（规则生成版）", "", "### 1. 当前窗口概览",
             f"- 时间窗口：{tw['start']} ~ {tw['end']}",
             f"- 记录数：产线 {tw['line_record_count']}，设备 {tw['device_record_count']}（L1={tw['l1_record_count']}，L2={tw['l2_record_count']}）", "",
             "### 2. L1 / L2 对比观察"]
    for item in top_diff:
        lines.append(f"- 观察到 {item['parameter']} 的 L2-L1 均值差={item['l2_minus_l1_mean']:.4g}，差值波动 std={item['diff_std']:.4g}，提示该参数值得进一步验证。")
    lines += ["", "### 3. 设备共有状态观察"]
    for item in top_device:
        lines.append(f"- 观察到 {item['parameter']} 波动较大（std={item['std']:.4g}，range={item['range']:.4g}），可能相关，建议下一步回放重点关注。")
    lines += ["", "### 4. 值得关注的参数",
              f"- L1 高波动参数：{', '.join([x['parameter'] for x in l1_vol]) or '无'}",
              f"- L2 高波动参数：{', '.join([x['parameter'] for x in l2_vol]) or '无'}",
              f"- 设备状态高波动参数：{', '.join([x['parameter'] for x in top_device]) or '无'}",
              "", "### 5. 下一步回放建议",
              f"- 建议参数：{', '.join(nxt.get('suggested_parameters', [])) or '无'}",
              f"- 建议设备参数：{', '.join(nxt.get('suggested_device_parameters', [])) or '无'}"]
    for view in nxt.get("suggested_views", []):
        lines.append(f"- 建议视图：{view}")
    lines += ["", "### 6. 说明", "本报告由规则生成，未调用大模型。当前结论基于设备 Log 的回放统计，不是最终因果结论，建议结合现场工艺经验和良率结果进一步确认。"]
    if error_message:
        lines += ["", f"> 模型调用失败原因：{error_message}"]
    return "".join(lines)


In [ ]:
line_window, device_window = filter_replay_window(
    line_state_long=line_state_long,
    device_state_df=device_state_df,
    start_time=REPLAY_START_TIME,
    end_time=REPLAY_END_TIME,
)

replay_summary = build_replay_summary(
    line_window=line_window,
    device_window=device_window,
    selected_params=SELECTED_REPLAY_PARAMS,
    selected_device_params=SELECTED_DEVICE_STATE_PARAMS,
    top_n=TOP_N_SUMMARY,
)

print(json.dumps(replay_summary, ensure_ascii=False, indent=2, default=str)[:6000])

prompt = build_ai_prompt(replay_summary)
ai_error = None
try:
    ai_report = call_llm(prompt)
except Exception as e:
    ai_report = None
    ai_error = str(e)

if not ai_report and USE_FALLBACK_REPORT:
    ai_report = build_fallback_report(replay_summary, error_message=ai_error)

if ai_report:
    display(Markdown(ai_report))
    print(ai_report)
else:
    print("未生成 AI 洞察报告。")

if SAVE_AI_INSIGHT_REPORT and ai_report:
    out_dir = PROCESSED_DIR / AI_INSIGHT_OUTPUT_DIR_NAME
    out_dir.mkdir(parents=True, exist_ok=True)
    start_label = pd.to_datetime(REPLAY_START_TIME).strftime("%Y%m%d_%H%M%S")
    end_label = pd.to_datetime(REPLAY_END_TIME).strftime("%Y%m%d_%H%M%S")
    out_path = out_dir / f"replay_insight_{start_label}_to_{end_label}.md"

    save_payload = {
        "time_window": replay_summary.get("time_window"),
        "selected_parameters": replay_summary.get("selected_parameters"),
        "selected_device_state_parameters": replay_summary.get("selected_device_state_parameters"),
        "top_line_diff_parameters": replay_summary.get("top_line_diff_parameters", [])[:10],
        "top_device_volatile_parameters": replay_summary.get("top_device_volatile_parameters", [])[:10],
        "next_replay_candidates": replay_summary.get("next_replay_candidates"),
    }

    report_text = f"""# AI 生产状态回放洞察

## 回放配置
- 开始时间：{REPLAY_START_TIME}
- 结束时间：{REPLAY_END_TIME}
- 观察参数：{SELECTED_REPLAY_PARAMS}
- 设备共有状态参数：{SELECTED_DEVICE_STATE_PARAMS}

## replay_summary 摘要
```json
{json.dumps(save_payload, ensure_ascii=False, indent=2, default=str)}
```

## AI / 规则洞察报告

{ai_report}
"""
    out_path.write_text(report_text, encoding="utf-8")
    print(f"AI 洞察报告已保存：{out_path}")
